# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates step-by-step how to load, explore, and analyze the [FAIR<sup>2</sup>] mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema available at:

<https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json>

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s for exploration.

In [ ]:
from pprint import pprint
record_sets_overview = []

print("Available record sets in the dataset:\n")
for rs in dataset.record_sets:
    record_set_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
    print(f"Record Set @id: {record_set_id}, Name: {getattr(rs, 'name', '')}")
    fields = getattr(rs, 'fields', [])
    # List fields and columns for each record set
    print("Fields and their @id's:")
    for field in fields:
        print(f"    Field @id: {getattr(field, '@id', None)} | Name: {getattr(field, 'name', '')} | DataType: {getattr(field, 'data_type', '')}")
    columns = getattr(rs, 'columns', [])
    if columns:
        print("Columns and their @id's:")
        for col in columns:
            print(f"    Column @id: {getattr(col, '@id', None)} | Name: {getattr(col, 'name', '')}")
    print('-'*40)
    record_sets_overview.append(record_set_id)

# Choose a record set @id to preview a sample of records
if dataset.record_sets:
    chosen_record_set_id = getattr(dataset.record_sets[0], '@id', None)
    print(f"\nSample records from record set: {chosen_record_set_id}\n")
    for i, rec in enumerate(dataset.records(record_set=chosen_record_set_id)):
        pprint(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load all record sets into pandas DataFrames for analysis. All lookups use each entity's `@id`.

In [ ]:
# List of all available record set @id's
record_set_ids = [getattr(rs, '@id', None) for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    recs = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(recs)

# Print available columns for first record set
if record_set_ids:
    print(f"Columns in DataFrame for {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group by field. All references are by `@id`.

In [ ]:
# For demonstration, select the most detailed/protein-based record set
# Find candidate numeric and group fields by inspecting the columns above

from numpy import number

main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Find numeric columns
numeric_candidates = df.select_dtypes(include='number').columns.tolist()
print(f"Numeric fields available for EDA: {numeric_candidates}")
if numeric_candidates:
    numeric_field = numeric_candidates[0]  # choose the first one for demo
    threshold = df[numeric_field].mean() if not df.empty else 0
    print(f"Using numeric field: {numeric_field}, threshold: {threshold:.2f}")

    # Filter records
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a likely group field, e.g. 'description' if it's a column
    possible_group_fields = [col for col in df.columns if col.lower().startswith('desc') or col.lower().startswith('group')]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (mean):")
        display(grouped_df.head())
    else:
        print("No obvious group field for grouping.")

## 5. Visualization
Visualize numeric field distributions and relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_candidates:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If 2+ numeric fields, show scatterplot
    if len(numeric_candidates) > 1:
        plt.figure(figsize=(6,5))
        sns.scatterplot(x=df[numeric_candidates[0]], y=df[numeric_candidates[1]])
        plt.title(f"{numeric_candidates[0]} vs {numeric_candidates[1]}")
        plt.show()

## 6. Conclusion

- We loaded and explored the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library.
- Record sets, fields, and columns were referenced by their Croissant `@id` properties, ensuring robust, schema-driven data handling.
- We performed filtering, normalization, grouping, and visualization of quantitative fields, illustrating a reproducible approach to FAIR scientific datasets.

Further analysis can be performed by adjusting the numeric and group fields, leveraging the full dataset metadata and schema.